In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/prakrutivara/prostructai/TS115_HHblits.npz
/kaggle/input/datasets/prakrutivara/prostructai/Train_HHblits.npz
/kaggle/input/datasets/prakrutivara/prostructai/CB513_HHblits.npz
/kaggle/input/models/prakrutivara/protein-ss-predictor/pytorch/default/1/__huggingface_repos__.json
/kaggle/input/models/prakrutivara/protein-ss-predictor/pytorch/default/1/best_model.pt


In [2]:
import os
import json
import math
import random
import time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [3]:
# ============================================================
# CONFIGURATION
# ============================================================

ESM_MODEL_NAME = "facebook/esm2_t12_35M_UR50D"

Q8_CLASSES = [
    "G", "H", "I", "B",
    "E", "S", "T", "C"
]

Q8_TO_Q3 = {
    "H": "H",
    "G": "H",
    "I": "H",

    "E": "E",
    "B": "E",

    "S": "C",
    "T": "C",
    "C": "C"
}

IGNORE_INDEX = -100

MAX_LEN = 400

BATCH_SIZE = 8

LSTM_HIDDEN = 256
LSTM_LAYERS = 2
DROPOUT = 0.3

# IMPORTANT:
# The saved successful model was trained with 3 unfrozen ESM layers.
N_UNFROZEN_LAYERS = 3

LR = 2e-4
ESM_LR = 2e-5

WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05

GRAD_ACCUM_STEPS = 1

EPOCHS = 25
PATIENCE = 5

OUT_DIR = "/kaggle/working"

CHECKPOINT_PATH = (
    "/kaggle/input/models/"
    "prakrutivara/protein-ss-predictor/"
    "pytorch/default/1/best_model.pt"
)

print("Configuration loaded.")
print("ESM:", ESM_MODEL_NAME)
print("Q8 classes:", Q8_CLASSES)
print("Unfrozen ESM layers:", N_UNFROZEN_LAYERS)
print("Checkpoint:", CHECKPOINT_PATH)

Configuration loaded.
ESM: facebook/esm2_t12_35M_UR50D
Q8 classes: ['G', 'H', 'I', 'B', 'E', 'S', 'T', 'C']
Unfrozen ESM layers: 3
Checkpoint: /kaggle/input/models/prakrutivara/protein-ss-predictor/pytorch/default/1/best_model.pt


In [4]:
print(
    "Checkpoint exists:",
    os.path.exists(CHECKPOINT_PATH)
)

if os.path.exists(CHECKPOINT_PATH):
    size_mb = os.path.getsize(CHECKPOINT_PATH) / (1024 ** 2)
    print(f"Checkpoint size: {size_mb:.2f} MB")

Checkpoint exists: True
Checkpoint size: 139.67 MB


In [5]:
# ============================================================
# LOAD + DECODE TRAIN / CB513 / TS115
# ============================================================

TRAIN_NPZ = "/kaggle/input/datasets/prakrutivara/prostructai/Train_HHblits.npz"
CB513_NPZ = "/kaggle/input/datasets/prakrutivara/prostructai/CB513_HHblits.npz"
TS115_NPZ = "/kaggle/input/datasets/prakrutivara/prostructai/TS115_HHblits.npz"

AA_ORDER = "ACDEFGHIKLMNPQRSTVWY"

# Q8 order used in the original data
Q8_CLASSES = ["G", "H", "I", "B", "E", "S", "T", "C"]


def decode_npz(path):
    
    d = np.load(path, allow_pickle=True)

    print("\nFILE:", path)
    print("Keys:", d.files)

    arr = d["data"]

    print("Raw shape:", arr.shape)
    print("Raw dtype:", arr.dtype)

    sequences = []
    labels = []

    for sample in arr:

        # ----------------------------------------
        # Amino-acid one-hot: columns 0:20
        # ----------------------------------------
        aa = sample[:, 0:20]

        # ----------------------------------------
        # Q8 one-hot: columns 57:65
        # ----------------------------------------
        ss = sample[:, 57:65]

        # Real residue rows
        real = aa.sum(axis=1) > 0

        aa_real = aa[real]
        ss_real = ss[real]

        # Decode amino-acid sequence
        aa_idx = aa_real.argmax(axis=1)

        seq = "".join(
            AA_ORDER[i]
            for i in aa_idx
        )

        # Decode Q8 labels
        ss_idx = ss_real.argmax(axis=1)

        ss_chars = "".join(
            Q8_CLASSES[i]
            for i in ss_idx
        )

        # Safety check
        if len(seq) != len(ss_chars):
            raise ValueError(
                f"Length mismatch: "
                f"sequence={len(seq)}, "
                f"SS={len(ss_chars)}"
            )

        sequences.append(seq)
        labels.append(ss_chars)

    return sequences, labels


# ============================================================
# LOAD
# ============================================================

print("Loading TRAIN...")
train_seqs, train_ss8 = decode_npz(TRAIN_NPZ)

print("\nLoading CB513...")
cb513_seqs, cb513_ss8 = decode_npz(CB513_NPZ)

print("\nLoading TS115...")
ts115_seqs, ts115_ss8 = decode_npz(TS115_NPZ)


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

print("Train :", len(train_seqs))
print("CB513 :", len(cb513_seqs))
print("TS115 :", len(ts115_seqs))

print("\nExample TRAIN:")
print("Sequence:", train_seqs[0][:80])
print("SS8     :", train_ss8[0][:80])
print("Length  :", len(train_seqs[0]), len(train_ss8[0]))

print("\nExample CB513:")
print("Sequence:", cb513_seqs[0][:80])
print("SS8     :", cb513_ss8[0][:80])

print("\nExample TS115:")
print("Sequence:", ts115_seqs[0][:80])
print("SS8     :", ts115_ss8[0][:80])

Loading TRAIN...

FILE: /kaggle/input/datasets/prakrutivara/prostructai/Train_HHblits.npz
Keys: ['pdbids', 'data']
Raw shape: (10848, 1632, 68)
Raw dtype: float64

Loading CB513...

FILE: /kaggle/input/datasets/prakrutivara/prostructai/CB513_HHblits.npz
Keys: ['pdbids', 'data']
Raw shape: (513, 874, 68)
Raw dtype: float64

Loading TS115...

FILE: /kaggle/input/datasets/prakrutivara/prostructai/TS115_HHblits.npz
Keys: ['pdbids', 'data']
Raw shape: (115, 1111, 68)
Raw dtype: float64

DATASET SUMMARY
Train : 10848
CB513 : 513
TS115 : 115

Example TRAIN:
Sequence: MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKRQT
SS8     : CCCCHHHHHHHHHHHHHHHHHHHHHHHCEEECCCCSEEETTSSCSCCTTTTCCCCEECCSSSTTCCEEECSCCTTHHHHH
Length  : 330 330

Example CB513:
Sequence: RTDCYGNVNRIDTTGASCKTAKPEGLSYCGVSASKKIAERDLQAMDRYKTIIKKVGEKLCVEPAVIAGIISRESHAGKVL
SS8     : CCCTTCCGGGSCCCCBCHHHHTTTTCSCCBHHHHHHHHHHTHHHHHTTHHHHHHHHHHHTSCHHHHHHHHHHHHGGGTTC

Example TS115:
Sequence: TRLSEILDQTTVLNDLKTV

In [6]:
# ============================================================
# TOKENIZER + DATASET + COLLATE
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    ESM_MODEL_NAME
)

Q8_TO_IDX = {
    c: i for i, c in enumerate(Q8_CLASSES)
}


class SSDataset(Dataset):

    def __init__(self, sequences, ss8_labels):
        self.sequences = sequences
        self.ss8_labels = ss8_labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.ss8_labels[idx]


def collate_fn(batch):

    seqs, labels = zip(*batch)

    # Tokenize WITHOUT imposing a fixed MAX_LEN.
    # Padding will use the longest sequence in THIS batch.
    enc = tokenizer(
        list(seqs),
        padding=True,
        truncation=False,
        return_tensors="pt"
    )

    batch_len = enc["input_ids"].shape[1]

    label_tensor = torch.full(
        (len(seqs), batch_len),
        IGNORE_INDEX,
        dtype=torch.long
    )

    for i, lab in enumerate(labels):

        q8_idx = [
            Q8_TO_IDX[c]
            for c in lab
        ]

        # CLS occupies position 0.
        # Residues therefore start at position 1.
        n = min(
            len(q8_idx),
            batch_len - 2
        )

        label_tensor[
            i,
            1:1+n
        ] = torch.tensor(
            q8_idx[:n],
            dtype=torch.long
        )

    return enc, label_tensor

config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [7]:
# ============================================================
# CB513 + TS115 LOADERS
# ============================================================

cb513_ds = SSDataset(
    cb513_seqs,
    cb513_ss8
)

ts115_ds = SSDataset(
    ts115_seqs,
    ts115_ss8
)

cb513_loader = DataLoader(
    cb513_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

ts115_loader = DataLoader(
    ts115_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True
)

print("CB513 batches:", len(cb513_loader))
print("TS115 batches:", len(ts115_loader))

CB513 batches: 65
TS115 batches: 15


In [8]:
enc, labels = next(iter(cb513_loader))

print("Input shape :", enc["input_ids"].shape)
print("Labels shape:", labels.shape)

Input shape : torch.Size([8, 451])
Labels shape: torch.Size([8, 451])


In [9]:
order = np.argsort(
    [len(s) for s in train_seqs]
)

VAL_FRACTION = 0.10

val_idx = set(
    order[
        ::int(1 / VAL_FRACTION)
    ].tolist()
)

tr_seqs = []
tr_ss8 = []

va_seqs = []
va_ss8 = []

for i, (s, l) in enumerate(
    zip(train_seqs, train_ss8)
):

    if i in val_idx:
        va_seqs.append(s)
        va_ss8.append(l)

    else:
        tr_seqs.append(s)
        tr_ss8.append(l)

print(
    f"Train: {len(tr_seqs)} | "
    f"Val: {len(va_seqs)}"
)

Train: 9763 | Val: 1085


In [10]:
train_ds = SSDataset(
    tr_seqs,
    tr_ss8
)

val_ds = SSDataset(
    va_seqs,
    va_ss8
)

cb513_ds = SSDataset(
    cb513_seqs,
    cb513_ss8
)

ts115_ds = SSDataset(
    ts115_seqs,
    ts115_ss8
)


train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

cb513_loader = DataLoader(
    cb513_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

ts115_loader = DataLoader(
    ts115_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

print(
    f"Train: {len(train_ds)} | "
    f"Val: {len(val_ds)} | "
    f"CB513: {len(cb513_ds)} | "
    f"TS115: {len(ts115_ds)}"
)

Train: 9763 | Val: 1085 | CB513: 513 | TS115: 115


In [11]:
class ESM_BiLSTM_SSPredictor(nn.Module):

    def __init__(
        self,
        esm_model_name,
        n_classes,
        lstm_hidden=256,
        lstm_layers=2,
        dropout=0.3,
        n_unfrozen_layers=3
    ):

        super().__init__()

        self.esm = AutoModel.from_pretrained(
            esm_model_name
        )

        esm_dim = self.esm.config.hidden_size

        # Freeze everything
        for p in self.esm.parameters():
            p.requires_grad = False

        # Robust across HF versions
        if hasattr(self.esm.encoder, "layer"):
            layers = self.esm.encoder.layer
        else:
            layers = self.esm.encoder.layers

        # Unfreeze last N layers
        for layer in layers[-n_unfrozen_layers:]:

            for p in layer.parameters():
                p.requires_grad = True

        # Final LayerNorm
        if hasattr(
            self.esm.encoder,
            "emb_layer_norm_after"
        ):

            for p in (
                self.esm.encoder
                .emb_layer_norm_after
                .parameters()
            ):
                p.requires_grad = True

        # BiLSTM
        self.lstm = nn.LSTM(
            input_size=esm_dim,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=(
                dropout
                if lstm_layers > 1
                else 0.0
            )
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            lstm_hidden * 2,
            n_classes
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        esm_out = self.esm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=False,
            return_dict=True
        )

        hidden = esm_out.last_hidden_state

        lengths = (
            attention_mask
            .long()
            .sum(dim=1)
            .cpu()
        )

        packed = (
            nn.utils.rnn.pack_padded_sequence(
                hidden,
                lengths,
                batch_first=True,
                enforce_sorted=False
            )
        )

        packed_out, _ = self.lstm(packed)

        lstm_out, _ = (
            nn.utils.rnn.pad_packed_sequence(
                packed_out,
                batch_first=True,
                total_length=hidden.shape[1]
            )
        )

        logits = self.classifier(
            self.dropout(lstm_out)
        )

        return logits

In [12]:
model = ESM_BiLSTM_SSPredictor(
    ESM_MODEL_NAME,
    n_classes=len(Q8_CLASSES),
    lstm_hidden=LSTM_HIDDEN,
    lstm_layers=LSTM_LAYERS,
    dropout=DROPOUT,
    n_unfrozen_layers=N_UNFROZEN_LAYERS
).to(device)

print("Model created.")

print(
    "ESM total params      :",
    f"{sum(p.numel() for p in model.esm.parameters()):,}"
)

print(
    "ESM trainable params  :",
    f"{sum(p.numel() for p in model.esm.parameters() if p.requires_grad):,}"
)

head_params_count = sum(
    p.numel()
    for n, p in model.named_parameters()
    if not n.startswith("esm.")
)

print(
    "BiLSTM + classifier   :",
    f"{head_params_count:,}"
)

model.safetensors:   0%|          | 0.00/136M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/209 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model created.
ESM total params      : 33,500,401
ESM trainable params  : 8,314,080
BiLSTM + classifier   : 3,092,488


In [13]:
print(
    "N_UNFROZEN_LAYERS =",
    N_UNFROZEN_LAYERS
)

layers = model.esm.encoder.layer

for i, layer in enumerate(layers):

    trainable = any(
        p.requires_grad
        for p in layer.parameters()
    )

    print(
        f"Layer {i}: "
        f"{'TRAINABLE' if trainable else 'FROZEN'}"
    )

N_UNFROZEN_LAYERS = 3
Layer 0: FROZEN
Layer 1: FROZEN
Layer 2: FROZEN
Layer 3: FROZEN
Layer 4: FROZEN
Layer 5: FROZEN
Layer 6: FROZEN
Layer 7: FROZEN
Layer 8: FROZEN
Layer 9: TRAINABLE
Layer 10: TRAINABLE
Layer 11: TRAINABLE


In [14]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(device)
model.eval()

print("====================================")
print("BEST CHECKPOINT LOADED")
print("====================================")
print("Epoch:", checkpoint.get("epoch"))
print(
    "Best validation Q3:",
    checkpoint.get("best_val_q3")
)

BEST CHECKPOINT LOADED
Epoch: 9
Best validation Q3: 0.8218954608584467


In [15]:
enc, labels = next(
    iter(cb513_loader)
)

input_ids = enc["input_ids"].to(device)
attention_mask = enc["attention_mask"].to(device)

with torch.no_grad():

    logits = model(
        input_ids,
        attention_mask
    )

print(
    "Input shape :",
    input_ids.shape
)

print(
    "Logits shape:",
    logits.shape
)

print(
    "Labels shape:",
    labels.shape
)

Input shape : torch.Size([8, 451])
Logits shape: torch.Size([8, 451, 8])
Labels shape: torch.Size([8, 451])


In [16]:
# ============================================================
# EVALUATION
# ============================================================

def evaluate_model(loader, name):

    model.eval()

    total_loss = 0.0
    n_batches = 0

    all_true = []
    all_pred = []

    criterion_eval = nn.CrossEntropyLoss(
        ignore_index=IGNORE_INDEX
    )

    with torch.no_grad():

        for enc, labels in loader:

            input_ids = enc["input_ids"].to(
                device,
                non_blocking=True
            )

            attention_mask = enc[
                "attention_mask"
            ].to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            with torch.amp.autocast(
                device_type="cuda",
                enabled=(device.type == "cuda")
            ):

                logits = model(
                    input_ids,
                    attention_mask
                )

                loss = criterion_eval(
                    logits.reshape(-1, logits.shape[-1]),
                    labels.reshape(-1)
                )

            total_loss += loss.item()
            n_batches += 1

            preds = logits.argmax(dim=-1)

            mask = labels != IGNORE_INDEX

            all_true.append(
                labels[mask].cpu().numpy()
            )

            all_pred.append(
                preds[mask].cpu().numpy()
            )

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)

    # -------------------------
    # Q8
    # -------------------------

    q8_acc = accuracy_score(
        y_true,
        y_pred
    )

    q8_macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro"
    )

    # -------------------------
    # Convert Q8 → Q3
    # -------------------------

    true_q8 = [
        Q8_CLASSES[i]
        for i in y_true
    ]

    pred_q8 = [
        Q8_CLASSES[i]
        for i in y_pred
    ]

    true_q3 = [
        Q8_TO_Q3[c]
        for c in true_q8
    ]

    pred_q3 = [
        Q8_TO_Q3[c]
        for c in pred_q8
    ]

    q3_acc = accuracy_score(
        true_q3,
        pred_q3
    )

    q3_macro_f1 = f1_score(
        true_q3,
        pred_q3,
        average="macro"
    )

    print("\n" + "=" * 55)
    print(name)
    print("=" * 55)

    print(
        f"Loss       : "
        f"{total_loss / n_batches:.4f}"
    )

    print(
        f"Q8 Acc     : "
        f"{q8_acc:.4f} ({q8_acc * 100:.2f}%)"
    )

    print(
        f"Q8 MacroF1 : "
        f"{q8_macro_f1:.4f}"
    )

    print(
        f"Q3 Acc     : "
        f"{q3_acc:.4f} ({q3_acc * 100:.2f}%)"
    )

    print(
        f"Q3 MacroF1 : "
        f"{q3_macro_f1:.4f}"
    )

    return {
        "loss": total_loss / n_batches,

        "q8_acc": q8_acc,
        "q8_macro_f1": q8_macro_f1,

        "q3_acc": q3_acc,
        "q3_macro_f1": q3_macro_f1,

        "y_true": y_true,
        "y_pred": y_pred,

        "true_q3": true_q3,
        "pred_q3": pred_q3,
    }

In [17]:
cb513_results = evaluate_model(
    cb513_loader,
    "CB513"
)

ts115_results = evaluate_model(
    ts115_loader,
    "TS115"
)


CB513
Loss       : 0.9151
Q8 Acc     : 0.6778 (67.78%)
Q8 MacroF1 : 0.4470
Q3 Acc     : 0.8117 (81.17%)
Q3 MacroF1 : 0.8072

TS115
Loss       : 0.8170
Q8 Acc     : 0.7189 (71.89%)
Q8 MacroF1 : 0.4694
Q3 Acc     : 0.8263 (82.63%)
Q3 MacroF1 : 0.8195


In [18]:
# ============================================================
# SAVE FINAL EVALUATION RESULTS
# ============================================================

import json

summary = {
    "model": "ESM-2 + BiLSTM",
    "esm_model": ESM_MODEL_NAME,
    "n_unfrozen_layers": N_UNFROZEN_LAYERS,

    "CB513": {
        "loss": float(cb513_results["loss"]),
        "q8_accuracy": float(cb513_results["q8_acc"]),
        "q8_macro_f1": float(cb513_results["q8_macro_f1"]),
        "q3_accuracy": float(cb513_results["q3_acc"]),
        "q3_macro_f1": float(cb513_results["q3_macro_f1"]),
    },

    "TS115": {
        "loss": float(ts115_results["loss"]),
        "q8_accuracy": float(ts115_results["q8_acc"]),
        "q8_macro_f1": float(ts115_results["q8_macro_f1"]),
        "q3_accuracy": float(ts115_results["q3_acc"]),
        "q3_macro_f1": float(ts115_results["q3_macro_f1"]),
    }
}

with open(
    "/kaggle/working/evaluation_results.json",
    "w"
) as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

{
  "model": "ESM-2 + BiLSTM",
  "esm_model": "facebook/esm2_t12_35M_UR50D",
  "n_unfrozen_layers": 3,
  "CB513": {
    "loss": 0.9151297037418072,
    "q8_accuracy": 0.6778099061804415,
    "q8_macro_f1": 0.44701320371911224,
    "q3_accuracy": 0.8117229375992455,
    "q3_macro_f1": 0.807243371554057
  },
  "TS115": {
    "loss": 0.8169538418451945,
    "q8_accuracy": 0.7188707381549515,
    "q8_macro_f1": 0.4694190213671606,
    "q3_accuracy": 0.826272622517454,
    "q3_macro_f1": 0.8195082352973916
  }
}
